# Retrieval v2 – Stage 0 / Stage 1

This notebook is a thin control and inspection layer.

- **Stage 0** independently reproduces the frozen Baseline v1 retrieval behavior and must pass the strict historical per-query fixture.
- **Stage 1** evaluates L0–L4 lexical retrieval on the four diagnostic ASR/text channels.
- Retrieval algorithms and metrics live in `src/`, not in notebook cells.
- Holdout20 is not evaluated here.


## 1. Setup

Mount Google Drive, resolve the code root and artifact root, and verify that the Retrieval v2 files are present.

Normally no path edit is needed when the project is stored under `aic26/asr_model_comparison`. Use the override variables only when running from a different location.


In [ ]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass


CODE_ROOT_OVERRIDE     = ""
ARTIFACT_ROOT_OVERRIDE = ""


def first_existing(candidates) :
    for candidate in candidates :
        path = Path(candidate)

        if (path.exists()) :
            return path.resolve()

    return None


def detect_artifact_root() :
    if (ARTIFACT_ROOT_OVERRIDE) :
        return Path(ARTIFACT_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_ARTIFACT_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        "/content/drive/MyDrive/aic26/asr_model_comparison",
    ]
    candidates.extend(
        glob.glob(
            "/content/drive/.shortcut-targets-by-id/*/aic26/asr_model_comparison"
        )
    )

    result = first_existing(candidates)

    if (result is None) :
        raise FileNotFoundError(
            "Could not detect the ASR artifact workspace. "
            "Set ARTIFACT_ROOT_OVERRIDE in this cell."
        )

    return result


def detect_code_root(artifact_root) :
    if (CODE_ROOT_OVERRIDE) :
        return Path(CODE_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_CODE_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        Path.cwd(),
        Path("/content/aic-2026-asr-retrieval"),
        artifact_root,
    ]

    for candidate in candidates :
        if ((candidate / "configs" / "retrieval_v2.json").exists()) :
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not detect the Retrieval v2 code root. "
        "Clone/copy the repository first or set CODE_ROOT_OVERRIDE."
    )


ARTIFACT_ROOT = detect_artifact_root()
CODE_ROOT     = detect_code_root(ARTIFACT_ROOT)

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CODE_ROOT"]     = str(CODE_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()


## 2. Stage selection

Run Stage 0 first. Stage 1 refuses to run unless the current Retrieval v2 config has already passed Stage 0.


In [ ]:
STAGE = "stage00_baseline"

# Use only:
# "stage00_baseline"
# "stage01_lexical"

assert STAGE in {"stage00_baseline", "stage01_lexical"}

print("Selected stage:", STAGE)


## 3. Configuration and integrity preview

This cell shows the frozen benchmark identities, channels, and retrieval policy before execution.


In [ ]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))

print("Retrieval ID :", config["retrieval_id"])
print("Frozen       :", config["retrieval_frozen"])
print()
print("Channels:")
for channel_id, item in config["channels"].items() :
    print(
        f"  {channel_id:20s} "
        f"model={item['model_id']:32s} "
        f"view={item['view']}"
    )

print()
print("Evaluation policy:")
print(json.dumps(config["evaluation_policy"], ensure_ascii = False, indent = 2))

print()
print("Stage 0 semantic model:")
print(json.dumps(config["stage00_baseline"]["semantic"], ensure_ascii = False, indent = 2))

print()
print("Strict Stage 0 fixture:")
print(json.dumps(config["regression"], ensure_ascii = False, indent = 2))


## 4. Execute Retrieval v2

The runner performs validation, retrieval, evaluation, caching, reporting, and manifest generation. A non-zero exit status is treated as a failed stage.


In [ ]:
command = [
    sys.executable,
    str(RUNNER),
    "--stage",
    STAGE,
]

print("Running:")
print(" ".join(command))
print()

completed = subprocess.run(
    command,
    cwd = str(CODE_ROOT),
    env = os.environ.copy(),
)

if (completed.returncode != 0) :
    raise RuntimeError(
        f"Retrieval v2 failed with exit code {completed.returncode}"
    )


## 5. Stage result

Load the machine-readable stage summary and validation report.


In [ ]:
REPORT_ROOT = ARTIFACT_ROOT / config["paths"]["reports_root"] / STAGE

summary_path    = REPORT_ROOT / "stage_summary.json"
validation_path = REPORT_ROOT / "validation_summary.json"

summary    = json.loads(summary_path.read_text(encoding = "utf-8"))
validation = json.loads(validation_path.read_text(encoding = "utf-8"))

print("=" * 88)
print("STAGE RESULT")
print("=" * 88)
print("Stage :", summary["stage"])
print("Status:", "PASS" if summary["passed"] else "FAIL")
print("Reports:", REPORT_ROOT)

if ("strict_fixture" in summary) :
    print()
    print("Strict historical fixture:")
    print(json.dumps(summary["strict_fixture"], ensure_ascii = False, indent = 2))

print()
print("Validation:", "PASS" if validation["passed"] else "FAIL")


## 6. Aggregate metrics

Stage 0 should reproduce the frozen Baseline v1 metrics. Stage 1 shows all five lexical methods independently for Whisper/Parakeet raw/processed.


In [ ]:
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")

metric_columns = [
    "query_set",
    "method_id",
    "model_id",
    "view",
    "query_count",
    "story_recall_at_1",
    "story_recall_at_3",
    "story_recall_at_5",
    "story_recall_at_10",
    "story_mrr",
    "video_recall_at_1",
    "video_recall_at_3",
    "video_recall_at_5",
    "video_recall_at_10",
    "video_recall_at_20",
    "video_mrr",
]

display(
    metrics[
        [column for column in metric_columns if column in metrics.columns]
    ].sort_values(
        ["query_set", "method_id", "model_id", "view"]
    ).reset_index(drop = True)
)


## 7. Per-query comparison

For Stage 0, inspect individual baseline failures or rank behavior.

For Stage 1, inspect L1–L4 against L0 and the paired method summary. The notebook does not automatically choose a winner.


In [ ]:
query_results = pd.read_csv(REPORT_ROOT / "query_results.csv")

if (STAGE == "stage01_lexical") :
    query_comparison = pd.read_csv(REPORT_ROOT / "query_comparison.csv")
    method_summary   = pd.read_csv(REPORT_ROOT / "method_summary.csv")

    print("Method-level paired summary:")
    display(method_summary)

    print()
    print("Per-query changes vs L0:")
    display(query_comparison.head(80))

else :
    print("Stage 0 query results:")
    display(
        query_results[
            [
                "query_set",
                "model_id",
                "view",
                "query_id",
                "first_relevant_rank",
                "video_rank",
                "top_story_window_id",
                "top_video_id",
                "story_score_margin",
                "video_score_margin",
            ]
        ].head(120)
    )


## 8. Failure and evidence inspection

Stage 1 stores normalized top evidence and lexical diagnostics. These preserve the window-level information needed for later temporal/video pooling research without changing Stage 1's max-window policy.


In [ ]:
if (STAGE == "stage01_lexical") :
    diagnostics = pd.read_csv(REPORT_ROOT / "lexical_diagnostics.csv")
    evidence    = pd.read_csv(REPORT_ROOT / "top_evidence.csv")

    print("Lowest lexical coverage / zero-evidence cases:")
    display(
        diagnostics.sort_values(
            ["zero_evidence", "query_coverage"],
            ascending = [False, True],
        ).head(40)
    )

    QUERY_ID = (
        "R2-20"
        if "R2-20" in set(evidence["query_id"])
        else evidence["query_id"].iloc[0]
    )

    print()
    print("Evidence for query:", QUERY_ID)
    display(
        evidence[
            evidence["query_id"] == QUERY_ID
        ].sort_values(
            ["method_id", "model_id", "view", "entity_type", "rank"]
        ).head(120)
    )

else :
    regression = json.loads(
        (REPORT_ROOT / "regression.json").read_text(encoding = "utf-8")
    )

    print("Stage 0 regression summary:")
    print(json.dumps(summary["regression"], ensure_ascii = False, indent = 2))

    if (not regression["passed"]) :
        print()
        print("First regression errors:")
        for error in regression.get("errors", [])[ : 20] :
            print(" -", error)


## 9. Research decision

Fill this only after reviewing Stage 1 metrics, paired query changes, coverage diagnostics, and representative failure cases.

The current roadmap does **not** fuse Whisper and Parakeet. Later retrieval stages will re-evaluate whether Parakeet becomes close enough to Whisper to keep only one ASR system.


In [ ]:
RETRIEVAL_DECISION = {
    "stage" : STAGE,
    "provisional_lexical_default" : None,
    "retained_alternative" : None,
    "main_observation" : None,
    "failure_observation" : None,
    "asr_gap_observation" : None,
    "notes" : [
        "Stage 1 selection is provisional because development20 contains only 20 queries.",
        "Whisper and Parakeet remain independent diagnostic channels; no ASR fusion is planned.",
        "Raw and processed remain diagnostic views; no text-view fusion is assumed.",
        "Holdout20 remains closed.",
    ],
}

print(
    json.dumps(
        RETRIEVAL_DECISION,
        ensure_ascii = False,
        indent = 2,
    )
)
